In [ ]:
# Compares two YOLO26m child detection checkpoints on the same data.

In [ ]:
# ============================================================
# INSTALLATION
# ============================================================

!pip install -q ultralytics pyyaml


# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path

import pandas as pd
import yaml
from ultralytics import YOLO


# ============================================================
# CHEMINS DES DEUX CHECKPOINTS
# ============================================================

OLD_CHECKPOINT = Path(
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "yolo26epoch100/best (2).pt"
)

NEW_CHECKPOINT = Path(
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "yolo26good/best (3).pt"
)


# ============================================================
# CHEMIN RÉEL DU DATASET YOLO
# ============================================================

DATASET_ROOT = Path(
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "childdetection/full_dataset/full_dataset"
)

TRAIN_IMAGES = DATASET_ROOT / "train/images"
TRAIN_LABELS = DATASET_ROOT / "train/labels"

VAL_IMAGES = DATASET_ROOT / "valid/images"
VAL_LABELS = DATASET_ROOT / "valid/labels"

TEST_IMAGES = DATASET_ROOT / "test/images"


# ============================================================
# VÉRIFICATIONS
# ============================================================

assert OLD_CHECKPOINT.exists(), (
    f"Ancien checkpoint introuvable : {OLD_CHECKPOINT}"
)

assert NEW_CHECKPOINT.exists(), (
    f"Nouveau checkpoint introuvable : {NEW_CHECKPOINT}"
)

assert TRAIN_IMAGES.exists(), (
    f"train/images introuvable : {TRAIN_IMAGES}"
)

assert TRAIN_LABELS.exists(), (
    f"train/labels introuvable : {TRAIN_LABELS}"
)

assert VAL_IMAGES.exists(), (
    f"valid/images introuvable : {VAL_IMAGES}"
)

assert VAL_LABELS.exists(), (
    f"valid/labels introuvable : {VAL_LABELS}"
)


print("Ancien checkpoint :", OLD_CHECKPOINT)
print("Nouveau checkpoint :", NEW_CHECKPOINT)
print("Dataset root       :", DATASET_ROOT)
print("Validation images  :", VAL_IMAGES)
print("Validation labels  :", VAL_LABELS)


# ============================================================
# CRÉATION D'UN DATA.YAML CORRIGÉ
# ============================================================

DATA_YAML = Path(
    "/kaggle/working/data_comparison_yolo.yaml"
)

yaml_content = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {
        0: "child",
    },
}

with DATA_YAML.open(
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        yaml_content,
        file,
        sort_keys=False,
    )

assert DATA_YAML.exists(), DATA_YAML

print("\nYAML corrigé créé :")
print(DATA_YAML)
print(DATA_YAML.read_text())


# ============================================================
# FONCTION D'ÉVALUATION
# ============================================================

def evaluate_checkpoint(
    checkpoint_path: Path,
    version_name: str,
) -> dict:

    print("\n" + "=" * 70)
    print(f"ÉVALUATION : {version_name}")
    print("=" * 70)
    print("Checkpoint :", checkpoint_path)

    model = YOLO(
        str(checkpoint_path)
    )

    metrics = model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=640,
        batch=8,
        device=0,
        workers=4,
        plots=False,
        verbose=True,
    )

    return {
        "Version": version_name,
        "Precision": float(
            metrics.box.mp
        ),
        "Recall": float(
            metrics.box.mr
        ),
        "mAP50": float(
            metrics.box.map50
        ),
        "mAP50-95": float(
            metrics.box.map
        ),
    }


# ============================================================
# ÉVALUATION ANCIEN CHECKPOINT
# ============================================================

old_results = evaluate_checkpoint(
    checkpoint_path=OLD_CHECKPOINT,
    version_name="Ancien checkpoint",
)


# ============================================================
# ÉVALUATION NOUVEAU CHECKPOINT
# ============================================================

new_results = evaluate_checkpoint(
    checkpoint_path=NEW_CHECKPOINT,
    version_name="Nouveau checkpoint",
)


# ============================================================
# TABLEAU COMPARATIF
# ============================================================

comparison_df = pd.DataFrame(
    [
        old_results,
        new_results,
    ]
)

metric_columns = [
    "Precision",
    "Recall",
    "mAP50",
    "mAP50-95",
]

comparison_df[metric_columns] = (
    comparison_df[metric_columns]
    .round(4)
)

print("\n" + "=" * 70)
print("COMPARAISON ANCIEN / NOUVEAU YOLO26m")
print("=" * 70)

display(
    comparison_df
)


# ============================================================
# DIFFÉRENCES
# ============================================================

print("\nÉvolution du nouveau checkpoint :")

for metric in metric_columns:
    difference = (
        new_results[metric]
        - old_results[metric]
    )

    sign = (
        "+"
        if difference >= 0
        else ""
    )

    print(
        f"{metric:10s} : "
        f"{sign}{difference:.4f}"
    )


# ============================================================
# SAUVEGARDE CSV
# ============================================================

OUTPUT_CSV = Path(
    "/kaggle/working/"
    "comparison_old_new_yolo26m.csv"
)

comparison_df.to_csv(
    OUTPUT_CSV,
    index=False,
)

print(
    "\nTableau sauvegardé dans :",
    OUTPUT_CSV,
)